In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession.builder
    .appName("TesteSpark")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/23 21:09:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
dados = [
    (1, "João", 5000),
    (2, "Maria", 8000),
    (3, "Pedro", 3000)
]

df = spark.createDataFrame(
    dados,
    ["id_cliente", "nome", "renda"]
)

df.show()

+----------+-----+-----+
|id_cliente| nome|renda|
+----------+-----+-----+
|         1| João| 5000|
|         2|Maria| 8000|
|         3|Pedro| 3000|
+----------+-----+-----+



In [4]:
df.printSchema()

root
 |-- id_cliente: long (nullable = true)
 |-- nome: string (nullable = true)
 |-- renda: long (nullable = true)



In [5]:
df.count()

3

In [7]:
df_filtrado = df.filter(df.renda > 4000)

In [9]:
df_filtrado.explain(True)

== Parsed Logical Plan ==
'Filter (renda#2L > 4000)
+- LogicalRDD [id_cliente#0L, nome#1, renda#2L], false

== Analyzed Logical Plan ==
id_cliente: bigint, nome: string, renda: bigint
Filter (renda#2L > cast(4000 as bigint))
+- LogicalRDD [id_cliente#0L, nome#1, renda#2L], false

== Optimized Logical Plan ==
Filter (isnotnull(renda#2L) AND (renda#2L > 4000))
+- LogicalRDD [id_cliente#0L, nome#1, renda#2L], false

== Physical Plan ==
*(1) Filter (isnotnull(renda#2L) AND (renda#2L > 4000))
+- *(1) Scan ExistingRDD[id_cliente#0L,nome#1,renda#2L]



In [10]:
df.rdd.getNumPartitions()

2

In [11]:
df.repartition(4).rdd.getNumPartitions()

4

In [12]:
df.repartition(4).explain(True)

== Parsed Logical Plan ==
Repartition 4, true
+- LogicalRDD [id_cliente#0L, nome#1, renda#2L], false

== Analyzed Logical Plan ==
id_cliente: bigint, nome: string, renda: bigint
Repartition 4, true
+- LogicalRDD [id_cliente#0L, nome#1, renda#2L], false

== Optimized Logical Plan ==
Repartition 4, true
+- LogicalRDD [id_cliente#0L, nome#1, renda#2L], false

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=94]
   +- Scan ExistingRDD[id_cliente#0L,nome#1,renda#2L]



In [13]:
spark.sparkContext.defaultParallelism

2

In [14]:
df.rdd.glom().collect()

[[Row(id_cliente=1, nome='João', renda=5000)],
 [Row(id_cliente=2, nome='Maria', renda=8000),
  Row(id_cliente=3, nome='Pedro', renda=3000)]]